[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vladimiracunadev-create/python-data-science-bootcamp/blob/master/classes/12-proyecto-final-y-cierre/notebook.ipynb)

# Clase 12 — Proyecto Final: Analisis Completo de Datos de Ventas

> **Este notebook integra todo lo aprendido en el bootcamp** en un analisis profesional completo.

---

## Que vas a construir

Un analisis end-to-end que un Data Scientist presentaria a un cliente real:

1. Exploracion de datos (EDA)
2. Limpieza y feature engineering
3. Analisis descriptivo de negocio
4. Modelo de regresion (predecir ventas)
5. Modelo de clasificacion (identificar transacciones de alto valor)
6. Conclusiones con numeros reales
7. Guardar el modelo para produccion


## Estructura de un proyecto profesional de Data Science

```
ESTRUCTURA DE UN PROYECTO DE DATA SCIENCE PROFESIONAL
=======================================================

Seccion 1: Contexto y pregunta de negocio
          |
          v
Seccion 2: Carga e inspeccion inicial de datos
          |  (shape, dtypes, nulls, describe, nunique)
          v
Seccion 3: Analisis Exploratorio (EDA)
          |  (histogramas, correlaciones, tendencias)
          v
Seccion 4: Limpieza y feature engineering
          |  (fechas, encoding, nuevas variables)
          v
Seccion 5: Modelado predictivo
          |  (regresion + clasificacion)
          v
Seccion 6: Conclusiones e insights
          |  (numeros reales, no vagas)
          v
Seccion 7: Recomendaciones para el negocio
             (acciones concretas basadas en evidencia)
```


In [ ]:
# ============================================================
# BLOQUE: Importaciones — todas las librerias del bootcamp
# ============================================================

# --- Manipulacion de datos ---
import pandas as pd    # DataFrames, groupby, fechas
import numpy as np     # arrays, operaciones matematicas

# --- Visualizacion ---
import matplotlib.pyplot as plt  # graficos base (barras, lineas, histogramas)
import seaborn as sns            # graficos estadisticos avanzados (heatmap, etc)

# --- Machine Learning: modelos ---
from sklearn.linear_model import LinearRegression   # regresion lineal para predecir ventas
from sklearn.linear_model import LogisticRegression # clasificacion binaria
from sklearn.tree import DecisionTreeClassifier     # arbol de decision

# --- Machine Learning: evaluacion ---
from sklearn.model_selection import train_test_split  # dividir datos
from sklearn.model_selection import cross_val_score   # validacion cruzada
from sklearn.model_selection import StratifiedKFold   # K-fold estratificado

# --- Machine Learning: preprocesamiento y pipeline ---
from sklearn.pipeline import Pipeline               # encadenar pasos de ML
from sklearn.preprocessing import StandardScaler    # escalar features

# --- Metricas ---
from sklearn.metrics import mean_absolute_error    # error absoluto medio (regresion)
from sklearn.metrics import mean_squared_error     # error cuadratico medio (regresion)
from sklearn.metrics import r2_score               # R2 (regresion)
from sklearn.metrics import classification_report  # precision, recall, f1 (clasificacion)
from sklearn.metrics import ConfusionMatrixDisplay # grafico de matriz de confusion
from sklearn.metrics import f1_score               # F1 individual

# --- Guardar/cargar modelos ---
import joblib  # serializar objetos Python: .dump() para guardar, .load() para cargar

# --- Utilidades ---
import warnings
warnings.filterwarnings("ignore")  # suprimir advertencias de deprecacion

# Configurar estilo visual de seaborn
sns.set_theme(style="whitegrid")   # fondo blanco con cuadricula gris
plt.rcParams["figure.dpi"] = 100  # resolucion de figuras

print("Todas las librerias del bootcamp cargadas correctamente")
print(f"  pandas:  {pd.__version__}")
print(f"  numpy:   {np.__version__}")


## Seccion 1: Contexto del proyecto


## Pregunta de negocio

> (Escribe aqui la pregunta que vas a responder con los datos)
> Ejemplo: "Que factores influyen mas en el total de ventas por transaccion?"

## Dataset

> `ventas_tienda.csv` — transacciones de una tienda.
> Variables esperadas: `fecha`, `sucursal`, `categoria`, `precio_unitario`, `unidades_vendidas`, `descuento_pct`, `total_venta`

## Objetivo del modelo

> **Regresion:** Predecir `total_venta` a partir de las caracteristicas de la transaccion.
> **Clasificacion:** Predecir si una transaccion sera de ventas altas (> mediana) o bajas.


In [ ]:
# ============================================================
# BLOQUE: Carga inicial e inspeccion del dataset
# ============================================================

# Paso 1: leer el archivo de ventas
# pd.read_csv infiere automaticamente los tipos de columna
df = pd.read_csv("../../datasets/ventas_tienda.csv")  # cargar datos de ventas

# Paso 2: dimensiones del dataset
print("=" * 50)
print("INSPECCION INICIAL DEL DATASET")
print("=" * 50)
print(f"  Filas:    {df.shape[0]}")
print(f"  Columnas: {df.shape[1]}")

# Paso 3: tipos de datos por columna
# .dtypes muestra el tipo (int64, float64, object) de cada columna
print("\nTipos de datos:")
print(df.dtypes)

# Paso 4: conteo de valores nulos
# .isnull().sum() devuelve el numero de NaN por columna
print("\nValores nulos por columna:")
nulos = df.isnull().sum()  # conteo de nulls
print(nulos[nulos > 0] if nulos.sum() > 0 else "  No hay valores nulos")

# Paso 5: estadisticas descriptivas de columnas numericas
# .describe() calcula count, mean, std, min, percentiles, max
print("\nEstadisticas descriptivas:")
print(df.describe().round(2))

# Paso 6: valores unicos por columna categorica
# .nunique() cuenta el numero de valores distintos en cada columna
print("\nValores unicos por columna:")
print(df.nunique())


In [ ]:
# ============================================================
# BLOQUE: Visualizacion de distribuciones
# ============================================================

# Paso 1: obtener la lista de columnas numericas
cols_num = df.select_dtypes(include=["number"]).columns.tolist()  # solo numericas
n_cols = len(cols_num)  # cuantas columnas numericas hay

# Paso 2: crear subgraficos dinamicos (ajustado al numero de columnas)
# math.ceil redondea hacia arriba para determinar el numero de filas
import math
ncols_plot = 3  # 3 graficos por fila
nrows_plot = math.ceil(n_cols / ncols_plot)  # filas necesarias

fig, axes = plt.subplots(nrows_plot, ncols_plot,
                          figsize=(14, nrows_plot * 4))  # alto dinamico
fig.suptitle("Distribuciones de Variables Numericas",
             fontsize=15, fontweight="bold", y=1.02)

# Paso 3: aplanar el array de ejes para iterar facilmente
axes_flat = axes.flatten()  # convertir 2D a 1D

# Paso 4: graficar cada columna numerica
for i, col in enumerate(cols_num):
    ax = axes_flat[i]  # eje correspondiente
    ax.hist(
        df[col].dropna(),    # datos sin nulos
        bins=40,             # 40 barras de frecuencia
        color="steelblue",   # color de las barras
        edgecolor="white"    # borde blanco entre barras
    )
    # Linea vertical de la media
    ax.axvline(df[col].mean(), color="red", linestyle="--",
               linewidth=1.5, label=f"Media={df[col].mean():.1f}")
    ax.set_title(col, fontsize=11)
    ax.set_ylabel("Frecuencia")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

# Paso 5: ocultar ejes sobrantes si el numero de columnas no es multiplo de ncols_plot
for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)  # ocultar eje no usado

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# BLOQUE: Heatmap de correlaciones
# ============================================================

# Paso 1: calcular la matriz de correlacion de Pearson
# .corr() calcula el coeficiente de correlacion entre todas las combinaciones de pares
# Valores: -1 = correlacion negativa perfecta, 0 = sin correlacion, 1 = positiva perfecta
corr_matrix = df[cols_num].corr(numeric_only=True)  # matriz cuadrada (n_cols x n_cols)

# Paso 2: crear mascara triangular superior (para no repetir informacion)
# np.triu_indices crea los indices del triangulo superior
mascara = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)  # triangulo superior True

# Paso 3: dibujar el heatmap
fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(
    corr_matrix,     # datos: matriz de correlacion
    mask=mascara,    # ocultar triangulo superior (datos repetidos)
    annot=True,      # mostrar el valor numerico en cada celda
    fmt=".2f",       # formato de 2 decimales
    cmap="RdYlGn",   # paleta: rojo (negativo) - amarillo (0) - verde (positivo)
    vmin=-1,         # minimo de la escala de color
    vmax=1,          # maximo de la escala de color
    center=0,        # centro de la escala en 0 (sin correlacion)
    square=True,     # celdas cuadradas para mejor visualizacion
    linewidths=0.5,  # lineas divisorias entre celdas
    ax=ax
)

ax.set_title("Heatmap de Correlaciones (Pearson)",
             fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

# Paso 4: identificar la feature mas correlacionada con total_venta
if "total_venta" in corr_matrix.columns:
    corr_con_target = corr_matrix["total_venta"].drop("total_venta").sort_values(key=abs, ascending=False)
    print("Correlacion con total_venta (ordenado por valor absoluto):")
    print(corr_con_target.round(3).to_string())


## Seccion 2: Limpieza y Feature Engineering


In [ ]:
# ============================================================
# BLOQUE: Limpieza y Feature Engineering completo
# ============================================================

# Paso 1: crear copia para no modificar el DataFrame original
# .copy() garantiza que los cambios no se propagen al original
df_clean = df.copy()  # copia independiente

# Paso 2: parsear la columna de fecha a tipo datetime
# pd.to_datetime convierte strings como '2023-01-15' a objetos datetime
# errors='coerce' convierte fechas invalidas a NaT en vez de lanzar error
df_clean["fecha"] = pd.to_datetime(df_clean["fecha"], errors="coerce")  # convertir a datetime
print(f"Tipo de columna 'fecha': {df_clean['fecha'].dtype}")

# Paso 3: extraer componentes temporales de la fecha
# .dt. da acceso al accessor de fechas de pandas
df_clean["mes"]          = df_clean["fecha"].dt.month        # 1=enero ... 12=diciembre
df_clean["dia_semana"]   = df_clean["fecha"].dt.dayofweek    # 0=lunes ... 6=domingo
df_clean["trimestre"]    = df_clean["fecha"].dt.quarter      # 1=Q1 ... 4=Q4

# Paso 4: crear feature binaria 'es_fin_de_semana'
# dia_semana >= 5 es True para sabado (5) y domingo (6)
# .astype(int) convierte True/False a 1/0
df_clean["es_fin_de_semana"] = (df_clean["dia_semana"] >= 5).astype(int)  # 1=fds, 0=entre semana

# Paso 5: manejar valores nulos en columnas numericas
# .fillna(mediana) imputa los nulos con la mediana (mas robusta que la media ante outliers)
for col in df_clean.select_dtypes(include=["number"]).columns:
    if df_clean[col].isnull().sum() > 0:  # solo si hay nulos en esta columna
        mediana = df_clean[col].median()   # calcular mediana ignorando nulos
        df_clean[col].fillna(mediana, inplace=True)  # imputar con mediana
        print(f"  Imputados {col} con mediana={mediana:.2f}")

# Paso 6: encodificar variables categoricas con one-hot encoding
# pd.get_dummies convierte columnas de texto en columnas binarias
# drop_first=True elimina una columna redundante por variable (evita multicolinealidad)
# Ejemplo: sucursal = [Norte, Sur, Este] -> sucursal_Sur, sucursal_Este (Norte es la referencia)
df_encoded = pd.get_dummies(
    df_clean.drop(columns=["fecha"]),      # eliminar fecha (ya extrajimos componentes)
    columns=["sucursal", "categoria"],      # columnas a encodificar
    drop_first=True,                        # eliminar primera categoria para evitar redundancia
    dtype=int                               # columnas de 0s y 1s (no bool)
)

print(f"\nDataset limpio y encodificado: {df_encoded.shape}")
print(f"Columnas nuevas: {list(df_encoded.columns)}")

# Verificar que no quedan nulos
print(f"\nNulos restantes: {df_encoded.isnull().sum().sum()}")


## Seccion 3: Analisis exploratorio


In [ ]:
# ============================================================
# BLOQUE: Top sucursales por total de ventas
# ============================================================

# Paso 1: agrupar por sucursal y sumar las ventas
# .groupby('sucursal') agrupa todas las filas de cada sucursal juntas
# ['total_venta'].sum() suma los valores de la columna total_venta por grupo
ventas_suc = (
    df_clean
    .groupby("sucursal")["total_venta"]  # agrupar por sucursal
    .sum()                               # sumar ventas de cada sucursal
    .sort_values(ascending=False)        # ordenar de mayor a menor
)

# Paso 2: tomar solo las 5 mejores sucursales
top5_suc = ventas_suc.head(5)  # primeras 5 despues de ordenar

# Paso 3: crear el grafico de barras
fig, ax = plt.subplots(figsize=(10, 5))

# Colorear la primera barra en dorado (la mejor sucursal)
colores = ["gold" if i == 0 else "steelblue" for i in range(len(top5_suc))]

# Dibujar las barras verticales
barras = ax.bar(
    top5_suc.index,    # nombres de sucursales en el eje X
    top5_suc.values,   # ventas totales en el eje Y
    color=colores,     # colores diferenciados
    edgecolor="white", # borde blanco entre barras
    width=0.6          # ancho de las barras
)

# Agregar etiquetas con el valor encima de cada barra
for barra in barras:
    altura = barra.get_height()  # valor de la barra
    ax.text(
        barra.get_x() + barra.get_width() / 2,  # posicion x: centro de la barra
        altura * 1.01,                           # posicion y: ligeramente encima
        f"${altura:,.0f}",                       # valor con formato de dinero
        ha="center", fontsize=10, fontweight="bold"
    )

ax.set_title("Top 5 Sucursales por Ventas Totales", fontsize=14, fontweight="bold")
ax.set_xlabel("Sucursal", fontsize=12)
ax.set_ylabel("Total Ventas ($)", fontsize=12)
ax.grid(True, alpha=0.3, axis="y")  # cuadricula solo horizontal

plt.tight_layout()
plt.show()

print("Ventas totales por sucursal:")
print(top5_suc.to_string())


In [ ]:
# ============================================================
# BLOQUE: Tendencia de ventas por mes
# ============================================================

# Paso 1: calcular ventas totales y promedio por mes
ventas_mes_total = (
    df_clean
    .groupby("mes")["total_venta"]  # agrupar por mes
    .sum()                          # ventas totales del mes
)
ventas_mes_prom = (
    df_clean
    .groupby("mes")["total_venta"]  # agrupar por mes
    .mean()                         # venta promedio por transaccion en ese mes
)

# Nombres cortos de meses para el eje X
nombres_meses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
                 "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

# Paso 2: crear figura con 2 subgraficos apilados
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))  # 2 filas, 1 columna

# Subgrafico 1: ventas totales (barras)
meses = ventas_mes_total.index.tolist()  # lista de meses presentes en los datos
ax1.bar(meses, ventas_mes_total.values, color="steelblue", edgecolor="white")
ax1.set_title("Ventas Totales por Mes", fontsize=13, fontweight="bold")
ax1.set_ylabel("Total Ventas ($)")
ax1.set_xticks(meses)
ax1.set_xticklabels([nombres_meses[m - 1] for m in meses])  # etiqueta de mes
ax1.grid(True, alpha=0.3, axis="y")

# Subgrafico 2: venta promedio por transaccion (linea con area rellena)
ax2.plot(meses, ventas_mes_prom.values, "o-", color="darkorange",
         linewidth=2.5, markersize=8, label="Promedio")
ax2.fill_between(meses, ventas_mes_prom.values,
                 alpha=0.2, color="darkorange")  # area bajo la curva
ax2.set_title("Venta Promedio por Transaccion y por Mes", fontsize=13, fontweight="bold")
ax2.set_xlabel("Mes")
ax2.set_ylabel("Venta Promedio ($)")
ax2.set_xticks(meses)
ax2.set_xticklabels([nombres_meses[m - 1] for m in meses])
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

# Identificar el mes de mayor venta
mes_top = ventas_mes_total.idxmax()  # mes con mayor total de ventas
print(f"Mes con mayores ventas totales: {nombres_meses[mes_top-1]} (mes {mes_top})")
print(f"  Total: ${ventas_mes_total[mes_top]:,.2f}")


In [ ]:
# ============================================================
# BLOQUE: Analisis por categoria (pastel + barras)
# ============================================================

# Paso 1: calcular ventas y numero de transacciones por categoria
ventas_cat = (
    df_clean
    .groupby("categoria")["total_venta"]  # agrupar por categoria
    .agg(["sum", "mean", "count"])         # tres metricas a la vez
    .rename(columns={"sum": "ventas_totales",
                     "mean": "venta_promedio",
                     "count": "n_transacciones"})
    .sort_values("ventas_totales", ascending=False)  # ordenar por total
)

# Paso 2: crear grafico de pastel + barras
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Pastel: distribucion porcentual de ventas por categoria
# autopct='%1.1f%%' muestra el porcentaje con 1 decimal
ax1.pie(
    ventas_cat["ventas_totales"],  # valores para el pastel
    labels=ventas_cat.index,       # etiquetas de categoria
    autopct="%1.1f%%",             # formato de porcentaje
    startangle=90,                 # empezar desde arriba
    colors=plt.cm.Set3.colors[:len(ventas_cat)]  # paleta de colores
)
ax1.set_title("Participacion por Categoria (% de Ventas)",
              fontsize=13, fontweight="bold")

# Barras: venta promedio por categoria
ax2.barh(
    ventas_cat.index,                     # categorias en eje Y
    ventas_cat["venta_promedio"],         # promedio en eje X
    color="teal", edgecolor="white"
)
ax2.set_title("Venta Promedio por Categoria", fontsize=13, fontweight="bold")
ax2.set_xlabel("Venta Promedio ($)")
ax2.grid(True, alpha=0.3, axis="x")

plt.tight_layout()
plt.show()

print("Resumen por categoria:")
print(ventas_cat.to_string())


## Seccion 4: Modelo Predictivo — Regresion

Vamos a predecir el `total_venta` de cada transaccion usando un modelo de regresion lineal.


In [ ]:
# ============================================================
# BLOQUE: Modelo de Regresion Lineal
# ============================================================

# Paso 1: definir features (X) y target de regresion (y)
# Excluir 'total_venta' del conjunto de features (es el target)
target_reg = "total_venta"  # variable a predecir (continua)
cols_reg = [c for c in df_encoded.columns if c != target_reg]  # features sin el target

X_reg = df_encoded[cols_reg]  # DataFrame de features
y_reg = df_encoded[target_reg]  # Serie con los valores a predecir

print(f"Features de regresion: {len(cols_reg)} columnas")
print(f"Target: {target_reg} (valor continuo)")

# Paso 2: dividir en train y test
# No usamos stratify porque el target es continuo (no hay clases)
X_tr, X_te, y_tr, y_te = train_test_split(
    X_reg, y_reg,
    test_size=0.2,   # 20% para evaluacion
    random_state=42  # reproducibilidad
)

print(f"\nTrain: {X_tr.shape[0]} filas")
print(f"Test:  {X_te.shape[0]} filas")

# Paso 3: crear y entrenar la regresion lineal
# LinearRegression ajusta los coeficientes w minimizando el error cuadratico
reg_lineal = LinearRegression()  # modelo de regresion lineal (y = w*X + b)
reg_lineal.fit(X_tr, y_tr)       # calcular los coeficientes optimos
print("\nModelo de regresion entrenado")

# Paso 4: predecir sobre el test set
y_pred_reg = reg_lineal.predict(X_te)  # array de predicciones (valores continuos)

# Paso 5: calcular las tres metricas principales de regresion
#
# MAE: error absoluto medio — cuantos $ nos equivocamos en promedio
#   MAE = mean(|y_real - y_pred|)
#
# RMSE: raiz del error cuadratico medio — penaliza errores grandes
#   RMSE = sqrt(mean((y_real - y_pred)^2))
#
# R2: coeficiente de determinacion — % de variabilidad explicada por el modelo
#   R2 = 1 - (varianza residual / varianza total)
#   R2 = 1.0 -> prediccion perfecta
#   R2 = 0.0 -> el modelo no es mejor que predecir siempre la media
#   R2 < 0   -> el modelo es peor que predecir la media

mae  = mean_absolute_error(y_te, y_pred_reg)        # error promedio en unidades del target
rmse = np.sqrt(mean_squared_error(y_te, y_pred_reg)) # error penalizando grandes equivocaciones
r2   = r2_score(y_te, y_pred_reg)                    # proporcion de varianza explicada

print("\nMETRICAS DEL MODELO DE REGRESION:")
print("=" * 45)
print(f"  MAE:  ${mae:,.2f}")
print(f"  RMSE: ${rmse:,.2f}")
print(f"  R2:   {r2:.4f} (explica el {r2*100:.1f}% de la variabilidad)")


## Seccion 5: Modelo de Clasificacion

Creamos un target binario para identificar transacciones de alto valor.


In [ ]:
# ============================================================
# BLOQUE: Clasificacion de ventas altas vs bajas
# ============================================================

# Paso 1: crear el target binario basado en la mediana
# La mediana divide el dataset exactamente en 2 mitades
# 50% de las transacciones seran clase 0, 50% clase 1
mediana_venta = df_encoded["total_venta"].median()  # umbral de decision
print(f"Mediana de total_venta: ${mediana_venta:,.2f}")

# Crear columna binaria: 1 si la venta supera la mediana, 0 si no
df_clf = df_encoded.copy()  # copia para no modificar df_encoded
df_clf["ventas_altas"] = (df_clf["total_venta"] > mediana_venta).astype(int)  # 1/0

print(f"\nBalance del target binario:")
print(df_clf["ventas_altas"].value_counts())  # deberia estar balanceado 50/50

# Paso 2: definir features y target para clasificacion
# IMPORTANTE: excluir total_venta del modelo
# Si lo incluimos, el modelo trivialmente sabe si la venta es alta o baja
cols_clf = [c for c in df_clf.columns
            if c not in ["total_venta", "ventas_altas"]]  # features sin los dos targets

X_clf = df_clf[cols_clf]       # features
y_clf = df_clf["ventas_altas"]  # target binario (0=baja, 1=alta)

print(f"\nFeatures de clasificacion: {len(cols_clf)}")

# Paso 3: dividir con stratify para preservar el balance de clases
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_clf, y_clf,
    test_size=0.2,
    random_state=42,
    stratify=y_clf   # importante: preservar 50/50 en train y test
)

# Paso 4: construir y entrenar el pipeline
# Pipeline evita data leakage: el scaler solo aprende del train de cada fold
pipeline_clf = Pipeline([
    ("scaler", StandardScaler()),         # normalizar features a media=0, std=1
    ("modelo", LogisticRegression(         # clasificador lineal con sigmoide
        max_iter=1000, random_state=42
    ))
])

pipeline_clf.fit(X_tr_c, y_tr_c)  # entrenar el pipeline completo
print("Pipeline de clasificacion entrenado")

# Paso 5: predecir y evaluar
y_pred_clf = pipeline_clf.predict(X_te_c)  # predicciones: 0 o 1

print("\nClassification Report:")
print("=" * 55)
print(
    classification_report(
        y_te_c, y_pred_clf,
        target_names=["Ventas Bajas", "Ventas Altas"]
    )
)

# Paso 6: visualizar la matriz de confusion
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_te_c, y_pred_clf,
    display_labels=["Ventas Bajas", "Ventas Altas"],
    cmap="Blues", ax=ax
)
ax.set_title("Matriz de Confusion — Clasificacion de Ventas",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## Seccion 6: Conclusiones

Completa esta plantilla con los numeros reales obtenidos en tu analisis:

---

## Hallazgos principales

1. (Escribe aqui el hallazgo 1 con numero real)
2. (Escribe aqui el hallazgo 2 con numero real)
3. (Escribe aqui el hallazgo 3 con numero real)

## Modelo de regresion
- MAE: ___
- R2: ___
- Interpretacion: ___

## Modelo de clasificacion
- F1 Score: ___
- Precision: ___
- Recall: ___

## Recomendaciones para el negocio
1. (Accion concreta basada en el dato 1)
2. (Accion concreta basada en el dato 2)

---


In [ ]:
# ============================================================
# BLOQUE: Guardar y cargar el modelo con joblib
# ============================================================

# joblib es la forma estandar de serializar modelos scikit-learn
# .dump(objeto, ruta) serializa el objeto a disco en formato .pkl (pickle)
# .load(ruta) deserializa el archivo y devuelve el objeto original

# Paso 1: guardar el modelo de regresion
ruta_reg = "modelo_regresion_ventas.pkl"  # nombre del archivo de salida
joblib.dump(reg_lineal, ruta_reg)  # serializar a disco
print(f"Modelo de regresion guardado en: {ruta_reg}")

# Paso 2: guardar el pipeline de clasificacion
ruta_clf = "pipeline_clasificacion_ventas.pkl"  # nombre del archivo de salida
joblib.dump(pipeline_clf, ruta_clf)  # serializar a disco
print(f"Pipeline de clasificacion guardado en: {ruta_clf}")

# Paso 3: verificar que se pueden cargar y hacer predicciones
modelo_reg_cargado  = joblib.load(ruta_reg)   # cargar el modelo de regresion
pipeline_clf_cargado = joblib.load(ruta_clf)  # cargar el pipeline de clasificacion

# Paso 4: hacer una prediccion de prueba con los modelos cargados
# Usar las primeras 3 filas del test set como ejemplo
pred_reg_prueba = modelo_reg_cargado.predict(X_te.iloc[:3])    # predicciones de ventas
pred_clf_prueba = pipeline_clf_cargado.predict(X_te_c.iloc[:3]) # predicciones de clase

print("\nVerificacion de modelos cargados desde disco:")
print(f"  Predicciones de regresion (3 filas): {pred_reg_prueba.round(2)}")
print(f"  Predicciones de clasificacion (3 filas): {pred_clf_prueba}")
print("\nModelos guardados y verificados correctamente.")
print("En produccion: cargar el modelo con joblib.load() y llamar .predict()")


---

## Felicitaciones. Completaste el Bootcamp Python para Data Science.

```
RESUMEN DEL BOOTCAMP
=====================

Clase 01  Python Fundamentos
Clase 02  Pandas y Limpieza de Datos
Clase 03  Visualizacion Exploratoria
Clase 04  Estadistica Descriptiva
Clase 05  Visualizacion con Matplotlib
Clase 06  Texto, Fechas y Transformaciones
Clase 07  Mini Proyecto Guiado
Clase 08  Presentacion de Hallazgos
Clase 09  Machine Learning Intro (Regresion Lineal)
Clase 10  Modelos Supervisados (Clasificacion)
Clase 11  Evaluacion y Pipelines (CV, GridSearch)
Clase 12  Proyecto Final (este notebook)

HABILIDADES ADQUIRIDAS:
  - Analisis y limpieza de datos con pandas
  - Visualizaciones profesionales con matplotlib y seaborn
  - Modelos predictivos con scikit-learn
  - Evaluacion robusta con cross-validation
  - Pipelines reproducibles y seguros
  - Comunicacion de resultados de negocio
  - Guardar y cargar modelos para produccion
```

**Proximos pasos:**
- Kaggle: participa en competencias para seguir practicando
- GitHub: sube tus notebooks y construye tu portafolio publico
- ML Avanzado: Random Forest, XGBoost, Redes Neuronales
- MLOps: aprende a desplegar modelos en produccion con FastAPI o Streamlit
